# Resultados: semáforo fijo vs agente RL

Análisis comparativo final para el reporte del proyecto.

**Prerrequisito:** haber entrenado el agente con `python rl/train.py`.

## Setup

In [ ]:
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from rl.evaluate import evaluar_baseline, evaluar_agente, comparar_e_imprimir, generar_graficas
from sim.intersection import SimuladorCruce
from sim.metrics import MetricsMonitor

MODELO_PATH = 'rl/models/best_model.zip'
USAR_DUMMY  = True   # cambiar a False con simulador calibrado

print('Setup listo')

## Evaluar baseline

In [ ]:
print('Evaluando baseline (semaforo fijo)...')
m_base = evaluar_baseline(n_episodios=3, usar_dummy=USAR_DUMMY)

## Evaluar agente entrenado

In [ ]:
print('Evaluando agente RL...')
m_rl = evaluar_agente(MODELO_PATH, n_episodios=3, usar_dummy=USAR_DUMMY)

## Tabla comparativa

In [ ]:
comparar_e_imprimir(m_base, m_rl)

## Gráficas para el reporte

In [ ]:
generar_graficas(m_base, m_rl, output_dir='data/validation')

## Gráfica comparativa de series de tiempo

Ver la evolución de las colas durante un episodio completo para cada escenario.

In [ ]:
# Correr un episodio completo de cada uno para graficar series de tiempo
from rl.environment import CruceEnv
from stable_baselines3 import PPO

env = CruceEnv()

# Baseline
obs, _ = env.reset()
while not env.sim.t >= 3600:
    obs, _, done, _, _ = env.step(0)
    if done: break
monitor_base = env.sim.monitor

# Agente RL
model = PPO.load(MODELO_PATH)
obs, _ = env.reset()
while not env.sim.t >= 3600:
    accion, _ = model.predict(obs, deterministic=True)
    obs, _, done, _, _ = env.step(int(accion))
    if done: break
monitor_rl = env.sim.monitor

MetricsMonitor.comparar(monitor_base, monitor_rl,
                         label_a='Semaforo fijo', label_b='Agente RL',
                         guardar=True, output_path='data/validation/comparacion_series.png')